__This will be a walkthrough of uploading sample data to the data management system for each of the five task types.__

1. Single Label Classification
2. Muli-Label Classification
3. Object detection
4. Semantic Segmentation
5. Instance Segmentation

Note, these are not training samples, they are examples on how to ingest data into the infrastructure described in the README. It shows off the expected input data format (consistent with Ground Truth output) and allows you to test the infrastructure for different label types. While I did create a logging client that returns logs in pandas DataFrame format (and show it off), I highly recommend simply using Athena in the AWS console for log and table visualization. The same holds true for step function flow to detect any failure points.

The following imports the programmatic API main entry point for interacting with the app. Everything is accessible through this client. Before this step, we must login to our account and gain credentials and permissions to interact with the infrastructure. Set up a user in identity center assigned to the account containing the CDK app, grant necessary permissions, and login in the terminal to gain temporary credentials. Ensure the profile you are using is in te local AWS conig file and points to the appropriate account, role, and AWS region.

In [11]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid
# from your previous login.
!aws sso login --profile {profile_name}

Attempting to automatically open the SSO authorization page in your default browser.
If the browser does not open, open the following URL:

https://oidc.us-east-1.amazonaws.com/authorize?response_type=code&client_id=fBuaSoXukP5ECyUWfUNTiXVzLWVhc3QtMQ&redirect_uri=http%3A%2F%2F127.0.0.1%3A51929%2Foauth%2Fcallback&state=bfd916e9-68c3-4670-8da7-2807f92a27c7&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=EnmmNAZUg_u_yNZAzQ3hrbKXKpcWzlM9gsyrnVDGlVw
Successfully logged into Start URL: https://d-906625d369.awsapps.com/start


In [12]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

2025-12-18 23:22:22,378 - INFO - Instantiating user instance.
2025-12-18 23:22:22,495 - INFO - Loading cached SSO token for myDefaultSession


RuntimeError: Missing or unreadable SSM region param /cvdms/cvdmsv1/region in region us-east-1: An error occurred (ParameterNotFound) when calling the GetParameter operation: 

In [ ]:
manifest_path = r"samples/single_label/single_label_truth_output.manifest"
label_type = "single-label"
job_summary = "Upload some sample COCO images."
data_source = "COCO"

upload_info = app.start_upload_job(manifest_path,
                                   label_type,
                                   job_summary = job_summary,
                                   data_source = data_source)

In [ ]:
job_id = upload_info.get('job_id')
upload_error = upload_info.get('error')

if job_id:
    print(f"Upload start was a success, job id is {job_id}")
else:
    print(f"Upload start was a failure, error is {upload_error}, see log file in cvdms_platform/api_logs")

Use the log interface to see the logs for this job id stored in a pandas DataFrame. It can take some time for logs to show up.

In [ ]:
log_info = app.get_logs_by_job_id(job_id)

if log_info.get('logs_df'):
    log_df = log_info['logs_df']
    print('First 10 logs are:')
    print(log_df.head(10))
else:
    log_retrieval_error = log_info.get('error')
    print(f'Error getting logs: {log_retrieval_error}')